# CS781 Phase 1 - generation for one model
Attach as inputs: the code dataset, the data dataset and **the training output of ONE model** (its `final.pt`). Turn **Internet ON** and make sure the Kaggle secret `HF` holds your Hugging Face token (same as the starter notebook).
Set `MODEL_TAG`, then **Save Version -> Save & Run All** (GPU T4).
1. downloads the official Test file from Hugging Face (only used in step 3)
2. decoding sweep on dev prompts (validation text only): greedy / temp 0.1 top-p 0.95 / greedy + repetition penalty 1.1 and 1.2, scored with **BERTScore F1** (the primary metric; falls back to ROUGE-L + BLEU-4 if bert_score is unavailable)
3. generates Test predictions with the best setting -> `/kaggle/working/sub_<tag>/submission.zip` (upload to Codabench)

In [2]:
import os, sys, json, re, glob, shutil, subprocess, time
DATA = "/kaggle/input/datasets/mrinaldevnath/cs781-legal-data"
CODE_SRC = "/kaggle/input/datasets/mrinaldevnath/cs781-p1-train"
WORK = "/kaggle/working/cs781/train"
MODEL_TAG = "d"            # a b c d e - only used for naming the output folder
TEST_PATH = None           # None = download from Hugging Face (needs Kaggle secret "HF")
N_DEV = 200                # dev prompts for the decoding sweep
BATCH = 32                 # lower to 16 if you see an out-of-memory error (model A/D use more)
FIXED = None               # set e.g. "--temperature 0" to skip the sweep and force one setting

def find(root, name):
    hits = glob.glob(os.path.join(root, "**", name), recursive=True)
    assert hits, f"{name} not found under {root}"
    return sorted(hits, key=len)[0]
DATA = os.path.dirname(find(DATA, "manifest.json"))
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(os.path.dirname(find(CODE_SRC, "train.py")), WORK)
os.chdir(WORK)
CKPT = find("/kaggle/input", "final.pt")
DEV = find(DATA, "dev_prompts.jsonl")
if TEST_PATH is None:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import hf_hub_download
    tok_ = UserSecretsClient().get_secret("HF_TOKEN")
    TEST_PATH = hf_hub_download(repo_id="Exploration-Lab/CS-781-Capstone", filename="Phase_1_Pretraining/Test/test.json",
                                repo_type="dataset", token=tok_)
# stratified dev subset (dev_prompts.jsonl is ordered by source, so [:n] alone would be all one source)
rows = [json.loads(l) for l in open(DEV, encoding="utf-8") if l.strip()]
srcs = sorted({r.get("src") for r in rows}); per = max(1, N_DEV // len(srcs))
sub = [r for s_ in srcs for r in [x for x in rows if x.get("src") == s_][:per]]
DEV = "/kaggle/working/dev_subset.jsonl"
open(DEV, "w", encoding="utf-8").write("\n".join(json.dumps(r) for r in sub) + "\n")
print("dev subset:", len(sub), "prompts from", srcs)
print("ckpt:", CKPT); print("dev :", DEV); print("test:", TEST_PATH)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)

subprocess.run("pip install -q bert_score", shell=True)

def run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = p.stdout + p.stderr
    print(out[-1500:])
    return p.returncode, out

test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

dev subset: 200 prompts from ['acts', 'hc', 'sc', 'trib']
ckpt: /kaggle/input/models/mrinaldevnath/cs781-capstone-model-d/pytorch/phase-1/1/run_d_gdn_mla/final.pt
dev : /kaggle/working/dev_subset.jsonl
test: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Test/test.json
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.8 MB/s eta 0:00:00


## 1. Decoding sweep on dev prompts

In [3]:
SETTINGS = {"greedy": "--temperature 0",
            "t0.1_p0.95": "--temperature 0.1 --top_p 0.95",
            "greedy_rep1.1": "--temperature 0 --rep_penalty 1.1",
            "greedy_rep1.2": "--temperature 0 --rep_penalty 1.2"}
if FIXED: SETTINGS = {"fixed": FIXED}
scores = {}
for name, opt in SETTINGS.items():
    rc, out = run(f"python generate.py --ckpt {CKPT} --data_root {DATA} --input {DEV} --batch {BATCH} --bertscore "
                  f"--out_dir /kaggle/working/dev_{name} {opt}")
    m = re.search(r"ROUGE-L F1 ([\d.]+)\s+BLEU-4 ([\d.]+)", out)
    assert m, f"dev generation failed for {name} (rc={rc})"      # a BERTScore failure alone is tolerated
    bs = re.search(r"BERTScore F1 ([\d.]+)", out)
    scores[name] = float(bs.group(1)) if bs else float(m.group(1)) + float(m.group(2))
    print(name, "ROUGE-L/BLEU-4", m.groups(), "BERTScore", bs.group(1) if bs else "n/a")
BEST = max(scores, key=scores.get)
print("scores (BERTScore F1, else ROUGE-L + BLEU-4):", scores, "-> best:", BEST)

/kaggle/working/dev_greedy/submission.zip; 200 predictions, words/prediction min 1 mean 122.7
DEV proxy: ROUGE-L F1 0.2094   BLEU-4 0.0806
DEV BERTScore F1 0.8123
sample: "Nickel-25 per cent Nickel-25 per cent Nickel-25 per cent Nickel-25 per cent Nickel-25 per cent Nickel-25 per cent Nickel-25 per cent Nickel-25 per cent Nickel-25 per cent Nickel-25 per cent Nickel-25 per cent Nickel-25 per cent Nickel-25 per cent Nickel-25 per cent Nickel-25 per cent Nickel-25 per 

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 2731.87it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED

## 2. Test predictions

In [4]:
OUT = f"/kaggle/working/sub_{MODEL_TAG}"
rc, out = run(f"python generate.py --ckpt {CKPT} --data_root {DATA} --input {TEST_PATH} --batch {BATCH} --out_dir {OUT} {SETTINGS[BEST]}")
assert rc == 0, "test generation failed"
pj = json.load(open(f"{OUT}/prediction.json"))
bad = [k for k, v in pj.items() if not v["text"].strip()]
print(len(pj), "predictions; empty:", len(bad)); print(os.listdir(OUT))
shutil.rmtree("/kaggle/working/cs781", ignore_errors=True)
for d in glob.glob("/kaggle/working/dev_*"): shutil.rmtree(d, ignore_errors=True)

generated 32/1000  (7s)
generated 64/1000  (14s)
generated 96/1000  (21s)
generated 128/1000  (28s)
generated 160/1000  (35s)
generated 192/1000  (42s)
generated 224/1000  (49s)
generated 256/1000  (56s)
generated 288/1000  (63s)
generated 320/1000  (70s)
generated 352/1000  (77s)
generated 384/1000  (84s)
generated 416/1000  (91s)
generated 448/1000  (98s)
generated 480/1000  (105s)
generated 512/1000  (112s)
generated 544/1000  (117s)
generated 576/1000  (124s)
generated 608/1000  (131s)
generated 640/1000  (138s)
generated 672/1000  (145s)
generated 704/1000  (152s)
generated 736/1000  (156s)
generated 768/1000  (163s)
generated 800/1000  (170s)
generated 832/1000  (177s)
generated 864/1000  (184s)
generated 896/1000  (191s)
generated 928/1000  (198s)
generated 960/1000  (205s)
generated 992/1000  (212s)
generated 1000/1000  (219s)
wrote /kaggle/working/sub_d/prediction.json and /kaggle/working/sub_d/submission.zip; 1000 predictions, words/prediction min 1 mean 125.0

1000 predictio